In [ ]:
# import libraries
import pandas as pd
import numpy as np
import urllib.request
import json
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# read the data
book_wish = pd.read_csv('Books - Wishlist.csv')
book_23 = pd.read_csv('Books - 2023.csv')
book_24 = pd.read_csv('Books - 2024.csv')
book_25 = pd.read_csv('Books - 2025.csv')

book_wish.shape, book_23.shape, book_24.shape, book_25.shape

((30, 6), (10, 11), (2, 11), (3, 11))

In [ ]:
# check book_wish first 5 rows
book_wish.head()

,Name,Author,Year,Language,Total Pages,Read
0,Записки изъ подполья (Notes from Underground),Фёдор Михайлович Достоевский (Fyodor Dostoevsky),1864,English,NaN,False
1,Brave New World,Aldous Huxley,1932,English,206.0,False
2,Fahrenheit 451,Ray Bradbury,1953,English,189.0,False
3,A Wizard of Earthsea,Ursula K. Le Guin,1968,English,201.0,False
4,"Women, Race & Class (Donne, razza e colore)",Angela Davis,1981,Italian,NaN,False


In [ ]:
# # check book_23 first 5 rows
book_23.head()

,Book title,Author,Year,Language,Month,Vote,Resource,Current Page,Total Page,Progress,Missing Pages
0,레몬 (Lemon),권여선 (Kwon Yeo-sun),2021,Italian,Jan,⭐⭐,Library,144,144,NaN,0
1,Brividi. Storie che non vi faranno dormire la ...,Elisa De Marco,2022,Italian,Feb,⭐⭐⭐⭐,Kobo,188,188,NaN,0
2,I'm Glad My Mom Died,Jennette McCurdy,2022,English,Mar,⭐⭐⭐⭐,Kobo,320,320,NaN,0
3,Normal People,Sally Rooney,2018,English,Apr,⭐⭐⭐⭐,Kobo,288,288,NaN,0
4,아몬드 (Almond),손원평 (Won-Pyung Sohn),2017,English,May,⭐⭐⭐⭐⭐,Kobo,272,272,NaN,0


In [ ]:
# # check book_24 first 5 rows
book_24.head()

,Book title,Author,Year,Language,Month,Vote,Resource,Current Page,Total Page,Progress,Missing Pages
0,The Silent Patient,Alex Michaelides,2019,English,Feb,⭐⭐⭐⭐,Bought,339,339,NaN,0
1,La libreria dei gatti neri,Piergiorgio Pulixi,2023,Italian,Jul,⭐⭐⭐,Bought,296,296,NaN,0


In [ ]:
# # check book_25 first 5 rows
book_25.head()

,Book title,Author,Year,Language,Month,Vote,Resource,Current Page,Total Page,Progress,Missing Pages
0,Crónica de una muerte anunciada,Gabriel García Márquez,1981,Spanish,Jan,⭐⭐⭐⭐⭐,Kobo,128,128,NaN,0.0
1,Cadáver exquisito,Agustina Bazterrica,2017,Spanish,March,NaN,Kobo,85,392,NaN,307.0
2,Life 3.0,Max Tegmark,2017,English,March,NaN,Bought,0,335,NaN,NaN


In [ ]:
# clean the data
def clean_read_data(df):
    df_clean = df.copy()
    # handle NaN values and count the exact character for stars
    df_clean['Rating'] = df_clean['Vote'].apply(
        lambda x: str(x).count('⭐') if pd.notna(x) else 3  # assume 3 stars if unrated
    )
    # rename 'Book title' to match wishlist 'Name'
    if 'Book title' in df_clean.columns:
        df_clean = df_clean.rename(columns={'Book title': 'Name'})
    return df_clean[['Name', 'Author', 'Rating']]

read_books = pd.concat([
    clean_read_data(book_23), 
    clean_read_data(book_24), 
    clean_read_data(book_25)
], ignore_index=True)


In [ ]:
# prepare wishlist books
wishlist_books = book_wish[['Name', 'Author']].copy()
wishlist_books['Rating'] = 0 # Placeholder for wishlist


In [ ]:
# combine all for feature fetching
all_books = pd.concat([read_books, wishlist_books], ignore_index=True)
print(f"Total read books: {len(read_books)}, Wishlist books: {len(wishlist_books)}")

In [ ]:
# 
def fetch_book_info(title, author):
    query = f"intitle:{title} inauthor:{author}".replace(' ', '+').encode('ascii', 'ignore').decode('ascii')
    url = f"https://www.googleapis.com/books/v1/volumes?q={query}"
    
    try:
        response = urllib.request.urlopen(url)
        data = json.loads(response.read())
        
        if 'items' in data and len(data['items']) > 0:
            info = data['items'][0]['volumeInfo']
            description = info.get('description', '')
            categories = info.get('categories', [])
            return f"{' '.join(categories)} {description}"
    except Exception as e:
        print(f"Failed to fetch {title}: {e}")
        
    return "" # Return empty if not found or error occurs

# fetch descriptions for all books (this might take a few minutes)
print("Fetching metadata from Google Books API...")
all_books['Content'] = ""
for idx, row in all_books.iterrows():
    print(f"Fetching: {row['Name']}")
    content = fetch_book_info(row['Name'], row['Author'])
    # combine title, author, and description into a single text document
    all_books.loc[idx, 'Content'] = f"{row['Name']} {row['Author']} {content}"
    time.sleep(1) # Be nice to the API

In [ ]:

# 1. Vectorize text features
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(all_books['Content'])

# 2. Split vectors back to read and wishlist
read_vectors = tfidf_matrix[:len(read_books)]
wishlist_vectors = tfidf_matrix[len(read_books):]

# 3. Create a User Profile based on highly-rated books
# We shift rating so that 1-2 stars are penalized or ignored, and 4-5 stars are rewarded
ratings = read_books['Rating'].values
weights = ratings - 2.5 
weights = np.where(weights < 0, 0, weights) # Only consider books rated 3 and above

# Compute weighted average of read vectors
user_profile = np.average(read_vectors.toarray(), weights=weights, axis=0)

# 4. Predict / Rank Wishlist Books
# Calculate cosine similarity between the user profile and all wishlist books
similarities = cosine_similarity([user_profile], wishlist_vectors.toarray())

# Add scores back to dataframe
wishlist_books = wishlist_books.copy()
wishlist_books['Recommendation_Score'] = similarities[0]

# 5. Display the Top 10 Recommended Books from Wishlist
recommended = wishlist_books.sort_values(by='Recommendation_Score', ascending=False)
print("\n
✨ Top 10 Book Recommendations for You ✨\n
")
display(recommended[['Name', 'Author', 'Recommendation_Score']].head(10))